In [1]:
import torch
import torch.nn as nn
import uproot
import awkward as ak
import numpy as np

In [2]:
# Load weights from data files using uproot
mc_file = uproot.open('/global/cfs/cdirs/m3246/ZjetOmnifold/data/slimmed_files/WithTracks_ZjetOmnifold_May19_MGPy8FxFxRew_syst_train_Mar1023.root')
mc_tree = mc_file['OmniTree']
pd_file = uproot.open('/global/cfs/cdirs/m3246/ZjetOmnifold/data/slimmed_files/WithTracks_ZjetOmnifold_Aug5_PseudoDataSRew_Mar21_1.root')
pd_tree = pd_file['OmniTree']

In [3]:
mc_weights = ak.to_numpy(mc_tree['weight'].array())
pd_weights = ak.to_numpy(pd_tree['weight'].array())

In [4]:
# Send weights to pytorch tensor
mc_weights = torch.tensor(mc_weights)
pd_weights = torch.tensor(pd_weights)

In [5]:
# Expand dimensions to match the shape of the output of the model
mc_weights = mc_weights.unsqueeze(1)
pd_weights = pd_weights.unsqueeze(1)

In [6]:
# Concatenate weights
weights = torch.cat((mc_weights, pd_weights), 0)

In [17]:
# Standardize weights
# standardized_mc_weights = mc_weights * (torch.sum(pd_weights) / torch.sum(mc_weights))
# standardized_pd_weights = pd_weights
# standardized_mc_weights /= torch.mean(standardized_mc_weights)
# standardized_pd_weights /= torch.mean(standardized_pd_weights)
mc_standard_factor = 2 * torch.sum(mc_weights) / len(weights)
pd_standard_factor = 2 * torch.sum(pd_weights) / len(weights)
standardized_mc_weights = mc_weights / mc_standard_factor
standardized_pd_weights = pd_weights / pd_standard_factor

In [18]:
print(torch.sum(standardized_mc_weights))
print(torch.sum(standardized_pd_weights))
print(torch.mean(standardized_mc_weights))
print(torch.mean(standardized_pd_weights))

tensor(835864.9375)
tensor(835865.)
tensor(0.5868)
tensor(3.3805)


In [19]:
# Concatenate standardized weights
standardized_weights = torch.cat((standardized_mc_weights, standardized_pd_weights), 0)

In [20]:
bce_loss = nn.BCELoss(reduction='none')

In [21]:
# Make tensors of predictions and targets
predictions = 0.5 * torch.ones(weights.shape)
targets = torch.cat((torch.zeros(mc_weights.shape), torch.ones(pd_weights.shape)), 0)

In [22]:
print(weights.shape)
print(predictions.shape)
print(targets.shape)

torch.Size([1671730, 1])
torch.Size([1671730, 1])
torch.Size([1671730, 1])


In [23]:
loss = bce_loss(predictions, targets)

In [24]:
# Weighted loss
weighted_loss = loss * weights
standardized_weighted_loss = loss * standardized_weights
mean_weighted_loss = torch.mean(weighted_loss)
mean_standardized_weighted_loss = torch.mean(standardized_weighted_loss)

In [25]:
print(torch.mean(loss))
print(mean_weighted_loss)
print(mean_standardized_weighted_loss)

tensor(0.6931)
tensor(0.2204)
tensor(0.6931)


In [26]:
print(torch.sum(standardized_mc_weights))
print(torch.sum(standardized_pd_weights))

tensor(835864.9375)
tensor(835865.)
